# Gold: modelo estrela

Este notebook transforma as tabelas Silver em dimensoes analiticas e na tabela fato `workspace.olist_gold.fato_vendas`, com granularidade de um item por pedido.

A fato combina pedidos, itens, avaliacoes e o primeiro pagamento de cada pedido. A coluna `delivery_status` classifica o cumprimento do prazo estimado.

**Ordem de execucao:** execute o Bronze e o Silver antes deste notebook.

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Gold - modelo estrela

# COMMAND ----------

from pyspark.sql import Window, functions as F

CATALOG = "workspace"
SILVER = f"{CATALOG}.olist_silver"
GOLD = f"{CATALOG}.olist_gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")


def save_gold(dataframe, table_name):
    dataframe.write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).saveAsTable(f"{GOLD}.{table_name}")
    print(f"✅ {GOLD}.{table_name}: {dataframe.count()} registros")


# Dimensoes
customers = spark.table(f"{SILVER}.customers")
dim_clientes = customers.select(
    "customer_id", "customer_city", "customer_state"
).dropDuplicates(["customer_id"])
save_gold(dim_clientes, "dim_clientes")

products = spark.table(f"{SILVER}.products")
dim_produtos = products.select(
    "product_id", "product_category_name"
).dropDuplicates(["product_id"])
save_gold(dim_produtos, "dim_produtos")

# Dominio fixo de notas validas, independente das avaliacoes observadas.
dim_avaliacoes = spark.range(1, 6).select(
    F.col("id").cast("int").alias("review_score")
).withColumn(
    "review_label", F.concat(F.lit("nota_"), F.col("review_score").cast("string"))
)
save_gold(dim_avaliacoes, "dim_avaliacoes")

orders = spark.table(f"{SILVER}.orders")
dim_tempo = orders.filter(F.col("order_purchase_timestamp").isNotNull()).select(
    F.to_date("order_purchase_timestamp").alias("purchase_date"),
    F.year("order_purchase_timestamp").alias("year"),
    F.month("order_purchase_timestamp").alias("month"),
    F.dayofmonth("order_purchase_timestamp").alias("day"),
    F.date_format("order_purchase_timestamp", "yyyy-MM").alias("year_month"),
).dropDuplicates(["purchase_date"])
save_gold(dim_tempo, "dim_tempo")

# Mantem apenas o primeiro pagamento por pedido para evitar duplicar itens.
payments = spark.table(f"{SILVER}.payments")
payment_window = Window.partitionBy("order_id").orderBy(
    F.col("payment_sequential").asc_nulls_last(), F.col("payment_type")
)
primary_payment = payments.withColumn(
    "row_number", F.row_number().over(payment_window)
).filter(F.col("row_number") == 1).select("order_id", "payment_type")

items = spark.table(f"{SILVER}.order_items")
reviews = spark.table(f"{SILVER}.reviews").select("order_id", "review_score")

fato_vendas = (
    orders.alias("o")
    .join(items.alias("i"), F.col("i.order_id") == F.col("o.order_id"))
    .join(reviews.alias("r"), F.col("r.order_id") == F.col("o.order_id"), "left")
    .join(primary_payment.alias("p"), F.col("p.order_id") == F.col("o.order_id"), "left")
    .filter(
        F.col("o.order_purchase_timestamp").isNotNull()
        & (
            F.col("o.order_delivered_customer_date").isNull()
            | (
                F.col("o.order_delivered_customer_date")
                >= F.col("o.order_purchase_timestamp")
            )
        )
    )
    .select(
        F.col("o.order_id"),
        F.col("i.order_item_id"),
        F.col("o.customer_id"),
        F.col("i.product_id"),
        F.col("r.review_score"),
        F.to_date("o.order_purchase_timestamp").alias("purchase_date"),
        F.col("p.payment_type"),
        F.col("i.price"),
        F.col("i.freight_value"),
        F.datediff(
            "o.order_delivered_customer_date", "o.order_purchase_timestamp"
        ).alias("delivery_days"),
        F.datediff(
            "o.order_delivered_customer_date", "o.order_estimated_delivery_date"
        ).alias("delivery_delay_days"),
        F.when(
            F.col("o.order_delivered_customer_date").isNull(), "nao_entregue"
        )
        .when(
            F.datediff(
                "o.order_delivered_customer_date", "o.order_estimated_delivery_date"
            )
            <= 0,
            "adiantado_ou_no_prazo",
        )
        .when(
            F.datediff(
                "o.order_delivered_customer_date", "o.order_estimated_delivery_date"
            )
            <= 7,
            "atraso_1_7_dias",
        )
        .otherwise("atraso_mais_de_7_dias")
        .alias("delivery_status"),
    )
)
save_gold(fato_vendas, "fato_vendas")

print("\n✅ GOLD LAYER COMPLETO!")


✅ workspace.olist_gold.dim_clientes: 99441 registros
✅ workspace.olist_gold.dim_produtos: 32951 registros
✅ workspace.olist_gold.dim_avaliacoes: 5 registros
✅ workspace.olist_gold.dim_tempo: 634 registros
✅ workspace.olist_gold.fato_vendas: 112650 registros

✅ GOLD LAYER COMPLETO!
